# Семинар 6. Процессы и Pool

Опора — лекция 6 и домашка по multiprocessing (`pi_estimate`,
`git_stats`, `multi_git`). Минимум теории, живые прогоны.

| # | Раздел |
|---|---|
| 1 | `Process`: pid, `join`, `__main__`-guard |
| 2 | Пулы: `ThreadPoolExecutor` и `multiprocessing.Pool` |
| 3 | Счёт против GIL: π тремя способами |
| 4 | Что передаётся: `pickle`, top-level функции |
| 5 | Порядок результатов: `map` против `as_completed` |
| 6 | Мелкие джобы: чанкование |
| 7 | minipipe: `Pool` для предсказаний |
| 8 | Подбор k: валидация и `Pool` |
| 9 | Бонус: поиск с ранним выходом |
| 10 | Start methods: spawn / fork / forkserver |

Красная нить: **почему в домашке worker-функции top-level, почему
`pool.map`, а не `submit`, и почему `git show` по чанкам, а не по SHA**.

In [ ]:
import multiprocessing as mp
import os
import random
import time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed

# В Jupyter spawn не жилец: дочерний процесс не может перечитать ноутбук.
# Демки на fork (наследует память). В обычных .py (вся ваша домашка)
# дефолт 3.14 на macOS — spawn, и он работает — при условии guard'а ниже.
CTX = mp.get_context("fork")

print("default start method:", mp.get_start_method(), "| ядер:", os.cpu_count())

## 1. Процесс вместо потока

Тот же API, что у потоков семинара 5: `start()`, `join()`. Разница —
у ребёнка **свой GIL и своя память**: ничего общего, кроме того, что
передали аргументами.

В `.py`-файлах запуск пула под spawn обязан лежать под
`if __name__ == "__main__":` — иначе ребёнок при старте перечитает файл
и запустит пул снова (и так до бесконечности). В ноутбуке это не нужно,
поэтому здесь guard нигде не пишем.

In [ ]:
def show(n: int) -> None:
    print(f"процесс {n}: pid={os.getpid()}, родитель {os.getppid()}")
    time.sleep(0.1)


procs = [CTX.Process(target=show, args=(i,)) for i in range(3)]
for p in procs:
    p.start()
for p in procs:
    p.join()
print("main: все процессы завершены")

## 2. Пулы: ThreadPoolExecutor и multiprocessing.Pool

**Пул** — заранее созданные воркеры + очередь задач: не платим за
spawn/thread-create на каждую задачу. В Питоне два рабочих пула,
похожих снаружи и устроенных по-разному.

### ThreadPoolExecutor (`concurrent.futures`)

- **Что:** пул потоков внутри одного процесса. Все потоки делят память
  и один GIL.
- **Задача = `Future`**: `submit(fn, *args)` кладёт работу в очередь и
  сразу возвращает `Future`; `future.result()` дожидается результата
  (можно `result(timeout=...)`), `future.done()` — статус без ожидания.
- **`pool.map(fn, iterable)`** — для «применить ко всем»: вернёт
  ленивый итератор результатов в порядке входа.
- **`with pool:`** = `shutdown(wait=True)`: очередь больше не
  принимается, дожидаемся всех воркеров. Без `with` — `pool.shutdown()`.
- **`max_workers`**: по умолчанию `min(32, cpu_count + 4)` — для I/O
  больше ядер нормально.
- **Когда:** I/O-bound — сеть, диск, sleep. На блокировке поток отпускает
  GIL, соседние работают.

### multiprocessing.Pool

- **Что:** пул отдельных процессов, у каждого свой GIL и своя память.
  Общение — только pickle-передача аргументов и результатов по каналам.
- **Запуск техник:** `map` (результаты в порядке входа), `imap`/
  `imap_unordered` (ленивые, второй — в порядке готовности),
  `apply`/`apply_async` (одна задача).
- **`chunksize`** в `map`: на сколько элементов резать iterable на
  одну передачу (см. раздел 6).
- **Функция и аргументы должны pickle'иться** → top-level функции
  (раздел 4).
- **Когда:** CPU-bound — чистый Питон и numpy-участки, удерживающие GIL.

### Общий интерфейс Executor

`ProcessPoolExecutor` — тот же API (`submit`/`map`/`with`), что и
`ThreadPoolExecutor`, но воркеры — процессы. Код ниже в разделе 3
меняет одну строчку (`ThreadPoolExecutor` → `Pool`) и получает
ускорение: это и есть механика `pi_estimate`.

In [ ]:
def fetch(url: str) -> str:
    """Эмуляция I/O-задачи: сеть/диск/sleep отпускают GIL."""
    time.sleep(0.2)
    return f"ответ {url}"


urls = [f"https://example.com/{i}" for i in range(4)]

# submit: Future'ы можно собирать в любом порядке и ждать по одному
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    futures = [pool.submit(fetch, u) for u in urls]
    by_submit = [f.result() for f in futures]
print(f"submit: {time.perf_counter() - t0:.2f} c (последовательно было бы ~0.8 c)")

# map: та же работа одной строкой, результат в порядке urls
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    by_map = list(pool.map(fetch, urls))
print(f"map:    {time.perf_counter() - t0:.2f} c")
assert by_submit == by_map

In [ ]:
def heavy(x: int) -> int:
    """CPU-bound работа: держит GIL, потоки её не распараллелят."""
    total = 0
    for i in range(200_000):
        total += i * i % 7
    return x + total


with CTX.Pool(processes=4) as pool:
    print("map:        ", pool.map(heavy, [1, 2, 3, 4]))
    print("imap:       ", list(pool.imap(heavy, [5, 6])))
    print("apply:      ", pool.apply(heavy, (7,)))
    print("apply_async:", pool.apply_async(heavy, (8,)).get())

# ThreadPoolExecutor для сравнения с тем же Future-интерфейсом
def classify(x: int) -> str:
    return "even" if x % 2 == 0 else "odd"


with ThreadPoolExecutor(max_workers=4) as t_pool:
    print("threads map:", list(t_pool.map(classify, range(6))))
with ProcessPoolExecutor(max_workers=4, mp_context=CTX) as p_pool:
    print("processes map:", list(p_pool.map(classify, range(6))))

## 3. Счёт против GIL: π тремя способами

Семинар 5 закончился на том, что потоки CPU-bound счёт не ускоряют.
Та же задача (Monte Carlo π, чистый Python, ноль I/O) — три запуска.
Первые два обязаны быть одинаковыми, третий — быстрее.
Это ровно функции из `pi_estimate`.

In [ ]:
N_POINTS = 10_000_000


def count_hits(n: int) -> int:
    rng = random.Random(20261016)
    hits = 0
    for _ in range(n):
        x, y = rng.random(), rng.random()
        hits += x * x + y * y <= 1.0
    return hits


def estimate_pi_serial(n: int) -> float:
    return 4 * count_hits(n) / n


def estimate_pi_threading(n: int, workers: int = 4) -> float:
    per, rem = divmod(n, workers)
    parts = [per + (1 if i < rem else 0) for i in range(workers)]
    with ThreadPoolExecutor(workers) as pool:
        hits = sum(pool.map(count_hits, parts))
    return 4 * hits / n


def estimate_pi_multiprocessing(n: int, workers: int | None = None) -> float:
    workers = workers or (os.cpu_count() or 4)
    per, rem = divmod(n, workers)
    parts = [per + (1 if i < rem else 0) for i in range(workers)]
    with CTX.Pool(workers) as pool:
        hits = sum(pool.map(count_hits, parts))
    return 4 * hits / n


for name, fn in [
    ("serial", estimate_pi_serial),
    ("threads", estimate_pi_threading),
    ("processes", estimate_pi_multiprocessing),
]:
    t0 = time.perf_counter()
    pi = fn(N_POINTS)
    dt = time.perf_counter() - t0
    print(f"{name:>10}: {dt:5.2f} c   pi ≈ {pi:.4f}")

## 4. Что передаётся: `pickle`, top-level функции

Пул не «вызывает» функцию в другом процессе — он **сериализует** её
имя и аргументы через pickle и отправляет по каналу. Вывод:

- функция должна быть **top-level** в импортируемом модуле
  (lambda, closure и методы pickle не отдаёт);
- аргументы и результаты тоже должны pickle'иться;
- именно поэтому в `git_stats` и `multi_git` worker'ы называются
  `_process_chunk` / `_run_one` и живут на верхнем уровне файла.

In [ ]:
# lambda не pickle'ится — пул падает, даже на fork:
try:
    with ProcessPoolExecutor(max_workers=2, mp_context=CTX) as pool:
        print(list(pool.map(lambda x: x * x, range(4))))
except Exception as exc:
    print(f"{type(exc).__name__}: {exc}")


# top-level функция того же ноутбука — ещё и импортировать не надо:
def square(x: int) -> int:
    return x * x


with ProcessPoolExecutor(max_workers=2, mp_context=CTX) as pool:
    print("top-level:", list(pool.map(square, range(6))))

## 5. Порядок результатов

`pool.map(fn, items)` возвращает результаты **в порядке `items`** —
независимо от того, в каком порядке задачи реально закончились.
`submit` + `as_completed` отдаёт в порядке завершения.

В `multi_git` тест на порядок (`results[i].repo == repos[i]`) падает,
если собирать через `as_completed`. Выбирай инструмент под контракт.

In [ ]:
def slow_square(x: int) -> int:
    time.sleep((8 - x) * 0.02)  # ранние элементы завершаются позже
    return x * x


items = list(range(8))
with ProcessPoolExecutor(max_workers=4, mp_context=CTX) as pool:
    print("map:         ", list(pool.map(slow_square, items)))
    futures = [pool.submit(slow_square, x) for x in items]
    print("as_completed:", [f.result() for f in as_completed(futures)])

## 6. Мелкие джобы: чанкуй

20 000 крошечных задач — это 20 000 pickle-передач: полезная работа
по наносекунде, накладные расходы — подавляющие. Собери задачи в
чанки и отдай пулу чанки: одна передача, много работы внутри.

Нюанс: `Pool.map` сам режет iterable на чанки (аргумент `chunksize`),
так что «по одной задаче» ниже приходится форсировать через
`chunksize=1`. При неоднородных джобах чанкование лучше делать явно —
как здесь и в `git_stats`.

В `git_stats` это тот же приём на один уровень глубже: **один
`git show` на чанк SHAs**, а не subprocess на каждый коммит —
приватный стресс-тест ловит per-SHA anti-паттерн по времени.

In [ ]:
def tiny(x: int) -> int:
    return x * x


def tiny_chunk(chunk: list[int]) -> list[int]:
    return [tiny(x) for x in chunk]  # работа внутри воркера, передача одна


items = list(range(200_000))

t0 = time.perf_counter()
with CTX.Pool() as pool:
    per_task = list(pool.map(tiny, items, chunksize=1))  # передача на КАЖДЫЙ элемент
t_one = time.perf_counter() - t0

n_chunks = os.cpu_count() or 4
size = (len(items) + n_chunks - 1) // n_chunks
chunks = [items[i : i + size] for i in range(0, len(items), size)]  # см. _chunk_shas
t0 = time.perf_counter()
with CTX.Pool() as pool:
    chunked: list[int] = []
    for part in pool.map(tiny_chunk, chunks):
        chunked.extend(part)
t_many = time.perf_counter() - t0

assert per_task == chunked
print(f"по задаче ({len(items):>6} передач): {t_one:5.2f} c")
print(f"чанками   ({len(chunks):>6} передач): {t_many:5.2f} c")

## 7. minipipe: Pool для предсказаний

Машинное обучение в двух словах, чтобы было понятно, что мы параллелим.
Задача — **классификация**: по описанию объекта угадать его метку
(здесь метки всего две: 0 или 1).

- **Выборка** — таблица чисел: строка = один объект, столбцы = **признаки**
  (замеры, по которым объекты сравнивают). Представьте таблицу писем,
  где столбцы — «доля слов со скидкой», «число ссылок», …, а последний
  столбец — метка: спам (1) или не спам (0). Ниже данные синтетические:
  8000 объектов по 40 признаков, метка = «сумма первых двух признаков
  положительна».
- **Обучение** (`fit`) — модель смотрит на train-часть и запоминает её.
  У k-NN обучение честное: просто запомнить все train-строки с метками.
- **Предсказание** (`predict`) — для нового объекта найти k=5 самых
  похожих train-объектов («соседей», ближайших по расстоянию между
  признаками) и взять метку большинства — голосование соседей.
  Для каждого тест-объекта считаются расстояния до ВСЕХ train-строк —
  матрица (test × train) — вот это чистый CPU и параллелится.
- **Accuracy** — доля угаданных меток: `accuracy(y_test, predicted)`.

**Разбиение train/test** — про него спрашивают первым делом на любом
собеседовании по ML. Идея: проверять модель надо на данных, которых она
при обучении не видела, — иначе оценка качества нечестная, как экзамен,
где билет заранее известен. `train_test_split` делает это в три шага:

1. **Перемешать** все индексы (`rng.permutation`). Зачем: в реальных
   данные часто упорядочены (например, все спам-письма идут подряд) —
   без перемешивания в test попал бы один класс. `seed` фиксирует
   случайность: при том же seed сплит воспроизводится побайтово.
2. **Отрезать** хвост: первые ~75% перемешанных индексов → train,
   остальные ~25% → test.
3. Части **не пересекаются** и вместе дают всю выборку — ни один объект
   не потерян и не встретился дважды.

Схема: `x` — выборка (строки = объекты), `y` — метки к ним; функция
возвращает четыре куска: `x_train, x_test, y_train, y_test` — те же
строки `x` и `y`, но разрезанные по перемешанным индексам.

Дальше — тот же приём, что в разделе 2: тестовую выборку режем на
чанки, предсказываем чанками в пуле, склеиваем. Сначала — разбиение
изнутри и почему test прячут от модели.

In [ ]:
import numpy as np

from minipipe.knn import KNN
from minipipe.metrics import accuracy
from minipipe.split import train_test_split

rng = np.random.default_rng(7)
# одна выборка: 8000 объектов, 40 признаков; метка — знак суммы первых двух
x = rng.normal(size=(8000, 40))
y = (x[:, 0] + x[:, 1] > 0).astype(np.int64)
print(f"выборка: {x.shape[0]} объектов по {x.shape[1]} признакам, метки: {np.unique(y).tolist()}")

# разбиение изнутри — train_test_split делает ровно это (minipipe/split.py)
splitter = np.random.default_rng(0)
order = splitter.permutation(len(x))  # шаг 1: перемешать ИНДЕКСЫ, не строки
n_test = int(len(x) * 0.25)           # шаг 2: отрезать хвост ~25%
test_index, train_index = order[:n_test], order[n_test:]
print(f"очередь индексов: {order[:8]} ... всего {len(order)}")

# шаг 3: части не пересекаются и в сумме дают всю выборку
assert set(train_index).isdisjoint(test_index)
assert len(train_index) + len(test_index) == len(x)

# функция возвращает те же куски, что ручная разметка по этим индексам
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, seed=0)
assert (x_test == x[test_index]).all() and (y_train == y[train_index]).all()
print(f"train: {x_train.shape[0]} строк, test: {x_test.shape[0]} строк — совпало")

# тот же seed → тот же сплит, другой seed → другой
assert (train_test_split(x, y, test_size=0.25, seed=0)[1] == x_test).all()
assert not (train_test_split(x, y, test_size=0.25, seed=42)[1] == x_test).all()

# зачем прятать test: модель, оцененная на своих же данных, врёт
cheater = KNN(k=1).fit(x_train, y_train)
print(f"accuracy на train (модель это видела): {accuracy(y_train, cheater.predict(x_train)):.3f}")
print(f"accuracy на test  (модель этого не видела): {accuracy(y_test, cheater.predict(x_test)):.3f}")

In [ ]:
def predict_chunk(args: tuple[KNN, np.ndarray]) -> np.ndarray:
    knn, chunk = args
    return knn.predict(chunk)


model = KNN(k=5).fit(x_train, y_train)

t0 = time.perf_counter()
predicted = model.predict(x_test)
t_serial = time.perf_counter() - t0
print(f"serial:   {t_serial:.2f} c, accuracy {accuracy(y_test, predicted):.3f}")

n_chunks = os.cpu_count() or 4
chunks = [c for c in np.array_split(x_test, n_chunks) if len(c)]
t0 = time.perf_counter()
with CTX.Pool(n_chunks) as pool:
    parts = pool.map(predict_chunk, [(model, c) for c in chunks])
predicted_par = np.concatenate(parts)
t_parallel = time.perf_counter() - t0

assert (predicted == predicted_par).all(), "параллельный predict изменил результат"
print(f"Pool x{n_chunks}: {t_parallel:.2f} c  (speedup {t_serial / t_parallel:.1f}x)")

## 8. Подбор k: валидация и Pool

У k-NN есть ручка настройки — **гиперпараметр** `k`: сколько соседей
голосуют. k=1 — модель запоминает выборку вместе с шумом; k=1000 —
голосует толпа и сглаживает всё подряд. Какое k лучшее? Перебрать
несколько и померить accuracy.

Перебирать по test **нельзя**: мы только что видели, что оценка на
увиденных данных завышена, а test хочется тронуть ОДИН раз — для
честного вердикта в самом конце. Поэтому train режем ещё раз: из 6000
train-строк выделяем 1500 на **валидацию** (validation). Итого:
4500 fit / 1500 val / 2000 test. На val выбираем k, на test — финальный
одиночный замер.

Параллелизм тут другой, чем в разделе 7. Там ОДИН predict резали на
чанки, а здесь **каждый k — самостоятельная задача**: своя модель, свой
`fit`, свой `predict`, своя accuracy. Задачи независимы → отдаём пулу
их целиком. Массивы маленькие (~2 МБ на задачу), поэтому пиклится каждая
задача вместе с аргументами — как в разделе 4. `map` держит порядок →
результаты складываем в dict k → accuracy, берём лучший k.

In [ ]:
# из train выделяем валидацию: на ней выбираем k, test трогаем один раз в конце
x_fit, x_val, y_fit, y_val = train_test_split(x_train, y_train, test_size=0.25, seed=1)
print(f"fit: {len(x_fit)}, val: {len(x_val)}, test: {len(x_test)}")


def eval_k(args: tuple[int, np.ndarray, np.ndarray, np.ndarray, np.ndarray]) -> tuple[int, float]:
    """Одна независимая задача: обучить k-NN с таким k, вернуть accuracy на val."""
    k, x_fit, y_fit, x_val, y_val = args
    predicted = KNN(k=k).fit(x_fit, y_fit).predict(x_val)
    return k, accuracy(y_val, predicted)


candidate_ks = [1, 3, 5, 7, 11, 21, 51]

t0 = time.perf_counter()
scores_serial = dict(eval_k((k, x_fit, y_fit, x_val, y_val)) for k in candidate_ks)
t_serial = time.perf_counter() - t0

n_workers = min(len(candidate_ks), os.cpu_count() or 4)
t0 = time.perf_counter()
with CTX.Pool(n_workers) as pool:
    scores = dict(pool.map(eval_k, [(k, x_fit, y_fit, x_val, y_val) for k in candidate_ks]))
t_parallel = time.perf_counter() - t0

assert scores == scores_serial, "параллельный перебор изменил оценки"
for k in candidate_ks:
    print(f"k={k:>3}: accuracy {scores[k]:.3f}")
print(f"serial {t_serial:.2f} c, Pool x{n_workers} {t_parallel:.2f} c (speedup {t_serial / t_parallel:.1f}x)")

best_k = max(scores, key=scores.get)
final_pred = KNN(k=best_k).fit(x_train, y_train).predict(x_test)
print(f"лучший k={best_k}, честная accuracy на test: {accuracy(y_test, final_pred):.3f}")

## 9. Бонус: поиск с ранним выходом

Поиск «спрятанного купона»: хеш известен, а число, которое его
породило, — нет. Перебираем кандидатов, пока хеш не совпадёт. Нужен
**первый** найденный результат — ждать всех бессмысленно.

`imap_unordered` отдаёт результаты по мере готовности; нашли —
`terminate()` глушит остальных. Тот же приём работает для любого
поиска «до первого попадания».

In [ ]:
import hashlib

# Купон спрятан так: хеш известен, x — нет. md5 от f"seminar6-{x}" — это
# быстрая лотерея: перебираем x, пока хеш не совпадет. Позиция купона
# зашита заранее (x = 6_543_210), поэтому демка воспроизводима.
TARGET_DIGEST = "a551a232ce335b1a1bc512135a5504d7"  # md5("seminar6-6543210")
RANGE = 20_000_000


def mine(args: tuple[int, int]) -> int | None:
    """Первый x из [start, stop), чей md5 от 'seminar6-x' равен TARGET."""
    start, stop = args
    for x in range(start, stop):
        if hashlib.md5(f"seminar6-{x}".encode()).hexdigest() == TARGET_DIGEST:
            return x
    return None


# serial: перебор с нуля, пока не найдём
t0 = time.perf_counter()
nonce_serial = None
for x in range(RANGE):
    if mine((x, x + 1)) is not None:
        nonce_serial = x
        break
t_serial = time.perf_counter() - t0
print(f"serial:  {t_serial:.2f} c, nonce {nonce_serial}")

# parallel: чанки поиска, результат — по мере готовности
n_chunks = os.cpu_count() or 4
size = RANGE // n_chunks
chunks = [(i * size, (i + 1) * size) for i in range(n_chunks)]
t0 = time.perf_counter()
with CTX.Pool(n_chunks) as pool:
    nonce_par = None
    for found in pool.imap_unordered(mine, chunks):
        if found is not None:
            nonce_par = found
            pool.terminate()  # нашли — глушим остальных
            break
t_parallel = time.perf_counter() - t0

print(f"Pool x{n_chunks}: {t_parallel:.2f} c, nonce {nonce_par} (speedup {t_serial / t_parallel:.1f}x)")
assert nonce_serial == nonce_par == 6_543_210, "найден не тот nonce"


## 10. Start methods

- **spawn** (дефолт 3.14 на macOS/Windows): чистый интерпретатор
  перечитывает файл. Отсюда `__main__`-guard и top-level worker'ы.
- **fork**: ребёнок — снимок памяти родителя. Быстро, работает в
  Jupyter, но опасен, если родитель уже запустил потоки
  (дедлок из лекции: fork копирует замок в занятом состоянии).
- **forkserver**: сервер-посредник форкается заранее, в чистом
  состоянии. Компромисс для Linux.

Ваша домашка — обычные `.py` под spawn: файл перечитывается, функции
импортируются, всё честно.

In [ ]:
print("в этой сессии:", mp.get_start_method())
print("доступные:   ", mp.get_all_start_methods())

## Итоги

| Раздел | Где пригодится в домашке |
|---|---|
| `Process` + guard | любой запуск пула из `.py` |
| π тремя способами | `pi_estimate` целиком |
| pickle, top-level | `_process_chunk` (git_stats), `_run_one` (multi_git) |
| `map` держит порядок | `multi_git`: `results[i].repo == repos[i]` |
| чанкование | `_chunk_shas` + один `git show` на чанк (git_stats) |
| Pool для предсказаний | вариант ускорения minipipe в курсовом |
| независимые задачи в Pool | перебор гиперпараметров (подбор k) |

Дальше — лекция 7: то же самое, только без процессов вообще.